In [ ]:
pip install folium

In [10]:
import pandas as pd

# 1. Load data dari spreadsheet
df_direktori = pd.read_excel('prelist_SE2026.xlsx', sheet_name='prelist_SE2026')
df_keyword = pd.read_excel('prelist_SE2026.xlsx', sheet_name='keyword')

# 2. Ambil dan bersihkan data dari kolom 'keyword' (yang ingin dimasukkan) 
# dan kolom 'exclude' (yang ingin dikeluarkan)
keywords_include = df_keyword['keyword'].dropna().astype(str).str.strip()

# Pastikan nama kolom di excel sesuai, misalnya 'exclude'
keywords_exclude = df_keyword['exclude'].dropna().astype(str).str.strip()

# 3. Buat pola regex untuk yang di-include
pola_include = r'\b(' + '|'.join(keywords_include) + r')\b'

# 4. Buat kondisi filtering
# Kondisi 1: Harus mengandung kata dari kolom 'keyword'
kondisi_include = df_direktori['nama_usaha'].str.contains(pola_include, case=False, na=False, regex=True)

# Cek apakah daftar exclude ada isinya (mencegah error regex jika kolom exclude kosong)
if not keywords_exclude.empty:
    # Buat pola regex untuk yang di-exclude
    pola_exclude = r'\b(' + '|'.join(keywords_exclude) + r')\b'
    
    # Kondisi 2: Mengandung kata dari kolom 'exclude'
    kondisi_exclude = df_direktori['nama_usaha'].str.contains(pola_exclude, case=False, na=False, regex=True)
    
    # Gabungkan kondisi: True di include DAN ( & ) False di exclude (~ meniadakan kondisi)
    hasil_filter = df_direktori[kondisi_include & ~kondisi_exclude]
else:
    # Jika kolom exclude kosong, cukup filter dengan kondisi include saja
    hasil_filter = df_direktori[kondisi_include]

# 5. Simpan hasilnya
hasil_filter.to_excel('hasil_filter_usaha.xlsx', index=False)

C:\Users\mirfaiss\AppData\Local\Temp\ipykernel_27168\4132334323.py:19: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  kondisi_include = df_direktori['nama_usaha'].str.contains(pola_include, case=False, na=False, regex=True)
C:\Users\mirfaiss\AppData\Local\Temp\ipykernel_27168\4132334323.py:27: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  kondisi_exclude = df_direktori['nama_usaha'].str.contains(pola_exclude, case=False, na=False, regex=True)


In [11]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# 1. Baca file batas wilayah (GeoJSON)
print("Membaca data peta Pasaman...")
peta_pasaman = gpd.read_file('Final_Kec_202511309.geojson')

# Filter hanya untuk Kabupaten PASAMAN (berdasarkan properti nmkab di GeoJSON)
peta_pasaman = peta_pasaman[peta_pasaman['nmkab'] == 'PASAMAN'].copy()

# 2. Baca data daftar usaha (Excel)
print("Membaca data prelist usaha...")
df_usaha = pd.read_excel('hasil_filter_usaha.xlsx')

# Pastikan tidak ada data koordinat yang kosong
df_usaha = df_usaha.dropna(subset=['latitude', 'longitude'])

# 3. Ubah koordinat Excel menjadi GeoDataFrame
geometri_titik = [Point(lon, lat) for lon, lat in zip(df_usaha['longitude'], df_usaha['latitude'])]
gdf_usaha = gpd.GeoDataFrame(df_usaha, geometry=geometri_titik)

# Samakan Sistem Referensi Koordinat (CRS) ke EPSG:4326
gdf_usaha.set_crs(epsg=4326, inplace=True)
peta_pasaman.to_crs(epsg=4326, inplace=True)

# 4. Melakukan Spatial Join (Pengecekan posisi sekaligus mengambil data kecamatan)
print("Melakukan pengecekan lokasi dan mapping kecamatan...")
# how='left' artinya semua data usaha tetap ada
# predicate='within' artinya mencari titik yang berada di DALAM poligon
gdf_hasil = gpd.sjoin(gdf_usaha, peta_pasaman[['kdkec', 'geometry']], how='left', predicate='within')

# 5. Pisahkan data berdasarkan hasil join
# Jika kolom 'kdkec' tidak kosong (notnull), berarti titik tersebut berada di dalam wilayah Pasaman
mask_di_dalam = gdf_hasil['kdkec'].notnull()

usaha_clean = gdf_hasil[mask_di_dalam].copy()
usaha_false = gdf_hasil[~mask_di_dalam].copy()

# Rename kolom 'kdkec' menjadi 'kecamatan' agar lebih rapi di Excel
usaha_clean = usaha_clean.rename(columns={'kdkec': 'kdkec'})

# 6. Simpan hasil ke Excel baru
print("Menyimpan hasil ke Excel...")

# Menghapus kolom 'geometry' dan 'index_right' (kolom bantuan dari sjoin) sebelum simpan
cols_to_drop = ['geometry', 'index_right']

usaha_clean.drop(columns=cols_to_drop, errors='ignore').to_excel('CLEAN_prelist_SE2026_ekraf.xlsx', index=False)
usaha_false.drop(columns=cols_to_drop, errors='ignore').to_excel('FALSE_prelist_SE2026_ekraf.xlsx', index=False)

print(f"\n--- Selesai! ---")
print(f"Data di DALAM Pasaman (CLEAN): {len(usaha_clean)} baris")
print(f"Data di LUAR Pasaman (FALSE): {len(usaha_false)} baris")

Membaca data peta Pasaman...
Membaca data prelist usaha...
Melakukan pengecekan lokasi dan mapping kecamatan...
Menyimpan hasil ke Excel...

--- Selesai! ---
Data di DALAM Pasaman (CLEAN): 1412 baris
Data di LUAR Pasaman (FALSE): 49 baris


In [12]:
import pandas as pd
import geopandas as gpd
import json

# --- 1. PROSES DATA KECAMATAN (DARI GEOJSON) ---
print("Membaca data GeoJSON Pasaman...")
peta_pasaman = gpd.read_file('Final_Kec_202511309.geojson')

# Filter hanya untuk Kabupaten PASAMAN
peta_pasaman = peta_pasaman[peta_pasaman['nmkab'] == 'PASAMAN'].copy()

# Ambil hanya kolom properties (hapus kolom geometry agar file .js tidak terlalu besar)
# Kita ambil semua kolom seperti gid, luas, idkec, nmkec, dll.
df_kecamatan = peta_pasaman.drop(columns='geometry')

# Ubah ke dalam format list of dictionaries
list_kecamatan = df_kecamatan.to_dict(orient='records')


# --- 2. PROSES DATA USAHA (DARI EXCEL) ---
file_excel = 'CLEAN_prelist_SE2026_ekraf.xlsx'
print("Membaca data Excel...")

# Pastikan kdkec dibaca sebagai string agar angka 0 di depan tidak hilang
df_usaha = pd.read_excel(file_excel, dtype={'kdkec': str})
df_usaha = df_usaha.dropna(subset=['latitude', 'longitude'])

# Ambil kolom yang dibutuhkan saja
list_usaha = df_usaha[['nama_usaha', 'latitude', 'longitude', 'kdkec', 'kategori', 'skala_usaha']].to_dict(orient='records')


# --- 3. SIMPAN KE FILE JAVASCRIPT ---
print("Menyusun file data_usaha.js...")

# Membuat konten JS dengan dua variabel berbeda
js_content = (
    f"// Data Batas Wilayah dan Properti Kecamatan\n"
    f"var dataKecamatan = {json.dumps(list_kecamatan, indent=2)};\n\n"
    f"// Data Titik Lokasi Usaha\n"
    f"var dataUsaha = {json.dumps(list_usaha, indent=2)};"
)

with open('data_usaha.js', 'w') as f:
    f.write(js_content)

print(f"Berhasil!")
print(f"- {len(list_kecamatan)} kecamatan dimasukkan ke 'dataKecamatan'")
print(f"- {len(list_usaha)} titik usaha dimasukkan ke 'dataUsaha'")

Membaca data GeoJSON Pasaman...
Membaca data Excel...
Menyusun file data_usaha.js...
Berhasil!
- 12 kecamatan dimasukkan ke 'dataKecamatan'
- 1412 titik usaha dimasukkan ke 'dataUsaha'
